In [1]:
# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_hub as hub

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Resizing, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, accuracy_score, f1_score
)

# ============================
# DATASET PATH
# ============================
dataset_path = "/kaggle/input/datasets/orofinadedamola/freshcheck-fruit-ripeness-classification-dataset/datasets/processed/tomato/train"

print("Dataset path exists:", os.path.exists(dataset_path))
print("Folders inside dataset path:", os.listdir(dataset_path))

# ============================
# SETTINGS
# ============================
img_size = (224, 224)   # keep 224 for new required models
batch_size = 16
epochs = 3

# ============================
# LOAD DATA
# ============================
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

class_names = list(train_data.class_indices.keys())
num_classes = train_data.num_classes

print("Classes:", class_names)
print("Number of classes:", num_classes)

# # ============================
# # OLD 3 CNN MODELS
# # ============================

def build_model_1():
    model = Sequential([
        Input(shape=(224,224,3)),
        Resizing(128,128),

        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_model_2():
    model = Sequential([
        Input(shape=(224,224,3)),
        Resizing(128,128),

        Conv2D(32, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_model_3():
    model = Sequential([
        Input(shape=(224,224,3)),
        Resizing(128,128),

        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(256, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# # ============================
# # NEW REQUIRED CNN MODELS
# # ============================

def build_alexnet():
    model = Sequential([
        Input(shape=(224,224,3)),
        Conv2D(96, (11,11), strides=4, activation='relu'),
        MaxPooling2D(3,3),

        Conv2D(256, (5,5), padding='same', activation='relu'),
        MaxPooling2D(3,3),

        Conv2D(384, (3,3), padding='same', activation='relu'),
        Conv2D(384, (3,3), padding='same', activation='relu'),
        Conv2D(256, (3,3), padding='same', activation='relu'),
        MaxPooling2D(3,3),

        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_inceptionresnetv2():
    base = tf.keras.applications.InceptionResNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(224,224,3)
    )
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=base.input, outputs=out)
    model.compile(
        optimizer=Adam(0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_densenet201():
    base = tf.keras.applications.DenseNet201(
        include_top=False,
        weights='imagenet',
        input_shape=(224,224,3)
    )
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=base.input, outputs=out)
    model.compile(
        optimizer=Adam(0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

import shutil
shutil.rmtree('/root/.cache/tfhub_modules', ignore_errors=True)

# ============================
# NEW REQUIRED ViT MODELS
# ============================

# ============================
# ViT (Stable Version)
# ============================
from tensorflow.keras import layers, models

def build_vit_b32():
    inputs = layers.Input(shape=(224,224,3))

    # Patch extraction (32x32)
    patches = layers.Conv2D(
        filters=64,
        kernel_size=32,
        strides=32,
        padding='valid'
    )(inputs)

    # Flatten patches
    x = layers.Reshape((-1, 64))(patches)

    # Simple Transformer Encoder (approx ViT)
    x = layers.LayerNormalization()(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.LayerNormalization()(x)

    # MLP Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ============================
# DeiT (Stable Version)
# ============================
from tensorflow.keras import layers, models

def build_deit_small():
    inputs = layers.Input(shape=(224,224,3))

    x = layers.Conv2D(64, 16, strides=16)(inputs)   # patch size 16
    x = layers.Reshape((-1, 64))(x)

    x = layers.LayerNormalization()(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# ============================
# Swin Transformer (Working)
# ============================
def build_swin_tiny():
    inputs = layers.Input(shape=(224,224,3))

    x = layers.Conv2D(96, 4, strides=4)(inputs)
    x = layers.LayerNormalization()(x)

    x = layers.Conv2D(96, 3, padding='same')(x)
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model
# ============================
# TRAIN + EVALUATE
# ============================

def train_and_evaluate(model, model_name):
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        verbose=1
    )

    val_data.reset()
    y_pred_probs = model.predict(val_data, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = val_data.classes

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"\n{model_name} Results")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1-Score :", f1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f"{model_name} Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f"{model_name} Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "Parameters": model.count_params()
    }

# ============================
# RUN ALL MODELS
# ============================

results = []

# # old
results.append(train_and_evaluate(build_model_1(), "CNN Model 1"))
results.append(train_and_evaluate(build_model_2(), "CNN Model 2"))
results.append(train_and_evaluate(build_model_3(), "CNN Model 3"))

# required new
results.append(train_and_evaluate(build_alexnet(), "AlexNet"))
results.append(train_and_evaluate(build_inceptionresnetv2(), "InceptionResNetV2"))
results.append(train_and_evaluate(build_densenet201(), "DenseNet201"))
results.append(train_and_evaluate(build_vit_b32(), "ViT-B/32"))
results.append(train_and_evaluate(build_deit_small(), "DeiT-Small"))
results.append(train_and_evaluate(build_swin_tiny(), "Swin Transformer Tiny"))

# ============================
# FINAL RESULTS
# ============================

results_df = pd.DataFrame(results)
print("\nFinal Comparison Table:")
print(results_df.sort_values(by="Accuracy", ascending=False))

results_df.to_csv("final_results.csv", index=False)
print("\nSaved results as final_results.csv")

ModuleNotFoundError: No module named 'numpy'

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Input(shape=(128, 128, 3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])